# Análisis Exploratorio con Pandas: guía para no perder en el TCG de Pokémon

**Autora:** Abril Gabriela de los Ángeles Sánchez Pérez
**Fecha:** Abril 2026
**Dataset:** [Pokemon.csv — Kaggle](https://www.kaggle.com/datasets/abcsds/pokemon)

%pip install matplotlib

## Pregunta 1 — Limpieza + Estadísticas descriptivas
¿Hay valores nulos en el dataset? ¿Cuál es el perfil estadístico de los stats?

In [29]:
# Paso 1. Importacion de librerías y carga del dataset
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('Pokemon.csv')
df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [30]:
# Paso 2. Exploracion de las dimensiones del dataset
print(df.shape)
print(df.dtypes)

(800, 13)
#              int64
Name          object
Type 1        object
Type 2        object
Total          int64
HP             int64
Attack         int64
Defense        int64
Sp. Atk        int64
Sp. Def        int64
Speed          int64
Generation     int64
Legendary       bool
dtype: object


In [31]:
# Paso 3. Detectar valores nulos
print(df.isnull().sum())

#               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype: int64


In [32]:
# Paso 4. Limpieza de nulos y su verificación
df ['Type 2'] = df ['Type 2'].fillna('None')
print("Nulos restantes de la limpieza:", df.isnull().sum())

Nulos restantes de la limpieza: #             0
Name          0
Type 1        0
Type 2        0
Total         0
HP            0
Attack        0
Defense       0
Sp. Atk       0
Sp. Def       0
Speed         0
Generation    0
Legendary     0
dtype: int64


In [33]:
# Paso 5. Estadísticas descriptivas de los pokemon
stats_cols=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
df[stats_cols].describe().round(1)

,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed
count,800.0,800.0,800.0,800.0,800.0,800.0
mean,69.3,79.0,73.8,72.8,71.9,68.3
std,25.5,32.5,31.2,32.7,27.8,29.1
min,1.0,5.0,5.0,10.0,20.0,5.0
25%,50.0,55.0,50.0,49.8,50.0,45.0
50%,65.0,75.0,70.0,65.0,70.0,65.0
75%,80.0,100.0,90.0,95.0,90.0,90.0
max,255.0,190.0,230.0,194.0,230.0,180.0


## Interpretacion - Pregunta 1
El unico campo con nulos era Type 2 (Contando con 386 de 800), lo cual tiene sentido porque la mayoría de los pokemon solo cuenta con un tipo. 
Para el ejercicio se rellenó estos datos con 'None'- 
Los stats promedio de los pocemons (Fila mean) oscila entre los 68 y 79 puntos, siendo Attack el que tiene el promedio mas alto (79.1) y Sp. Def el que tiene el promedio mas bajo (68.3).

# Pregunta 2 - Transformación + Agrupaciones
¿Qué tipo de pókemon primario (Type 1) domina en términos de poder ofensivo? ¿Podría ser aquellos de tipo Dragon o los de tipo Pelea (Fighting)?
**Se analizara creando una nueva varibale donde Offensive_Power = Attack + Sp.Atk y se analizará su promedio por tipo**

In [34]:
# Paso 6. Creacion columna Offensive_Power
df['Offensive_Power'] = df['Attack'] + df['Sp. Atk']
df[['Name', 'Offensive_Power']].head()

,Name,Offensive_Power
0,Bulbasaur,114
1,Ivysaur,142
2,Venusaur,182
3,VenusaurMega Venusaur,222
4,Charmander,112


In [35]:
# Paso 7. Agrupar los pokemon Type 1
top_ofensivos = (
    df.groupby('Type 1')['Offensive_Power']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
top_ofensivos

,Type 1,Offensive_Power
0,Dragon,208.968750
1,Fire,173.750000
2,Flying,173.000000
3,Psychic,169.859649
4,Dark,163.032258
5,Steel,160.222222
6,Electric,159.113636
7,Rock,156.204545
8,Ghost,153.125000
9,Ground,152.218750


In [36]:
print ("🥇Ranking de Tipos de Pokemon más Ofensivos:\n")
for i, row in top_ofensivos.iterrows():
    print(f"{i+1}. {row['Type 1']:<12} → {round(row['Offensive_Power'], 1)}")

🥇Ranking de Tipos de Pokemon más Ofensivos:

1. Dragon       → 209.0
2. Fire         → 173.8
3. Flying       → 173.0
4. Psychic      → 169.9
5. Dark         → 163.0
6. Steel        → 160.2
7. Electric     → 159.1
8. Rock         → 156.2
9. Ghost        → 153.1
10. Ground       → 152.2
11. Grass        → 150.7
12. Ice          → 150.3
13. Fighting     → 149.9
14. Water        → 149.0
15. Fairy        → 140.1
16. Poison       → 135.1
17. Normal       → 129.3
18. Bug          → 124.8


## Interpretación — Pregunta 2

**Dragon domina con 209 puntos de poder ofensivo**, casi 35 puntos más que Fire en segundo lugar.

Esto explica varias cosas del universo Pokémon:

- El equipo Rocket siempre pierde porque usa mayormente Poison y Normal, 
  los tipos más bajos del ranking. ¡Con razón Ash los derrota con un Pikachu!
- Fighting está arriba (#12) pero su poder ofensivo es más físico (Attack),
  mientras que Psychic (#4) compensa con Sp. Atk — por eso Alakazam le gana a Machamp.

**Conclusión:** Si armas tu equipo con Dragon + Fire + Psychic, 
vas a destruir cualquier liga. Cynthia lo sabía desde siempre. 🏅

## Pregunta 3 — Filtrado + Visualización
¿Los Pokémon con un Total superior al promedio son mayormente legendarios?

In [37]:
# Paso 9. Calcular promedio y filtrar
promedio_total = df['Total'].mean()
print(f"Promedio de Total: {promedio_total:.1f}")

df_top = df[df['Total'] > promedio_total]
print(f"Pokémon sobre el promedio: {len(df_top)}")

Promedio de Total: 435.1
Pokémon sobre el promedio: 417


In [38]:
#Paso 10. Conteo de Legendarios
conteo = df_top['Legendary'].value_counts()
porcentaje = df_top['Legendary'].value_counts(normalize=True) * 100

print("\nConteo:")
print(conteo)
print("\nPorcentaje:")
print(porcentaje.round(1))


Conteo:
Legendary
False    352
True      65
Name: count, dtype: int64

Porcentaje:
Legendary
False    84.4
True     15.6
Name: proportion, dtype: float64


In [39]:
# Paso 11. Print de las estadísticas finales
print("¿Los pokémon fuertes son legendarios?\n")
for val, pct in zip(conteo.index, porcentaje.round(1)):
    etiqueta = "Legendario" if val else "No legendario"
    print(f"{etiqueta:<15} → {conteo[val]} pokémon ({pct}%)")

¿Los pokémon fuertes son legendarios?

No legendario   → 352 pokémon (84.4%)
Legendario      → 65 pokémon (15.6%)


## Interpretación — Pregunta 3 

El promedio de Total en todo el dataset es **435.1 puntos**.
417 pokémon lo superan — o sea más de la mitad son "fuertes".

De esos 417 pokémon fuertes:
- No legendarios → 352 (84.4%)
- Legendarios → 65 (15.6%)

**Conclusión:** Ser legendario NO es requisito para ser fuerte. 
El 84.4% de los pokémon poderosos son completamente normales (bueno, normales no, 
porque Normal como tipo está en el fondo del ranking).

Esto significa que si entrenas bien a tu equipo — Dragonite, Garchomp, 
Tyranitar — puedes vencer a Mewtwo sin necesitar un pokémon legendario.

**Dato curioso:** Solo el 15.6% son legendarios, pero en el imaginario 
colectivo sentimos que son imbatibles. Puro marketing del profesor Oak.

## Pregunta 4 - ¿Cuál es el equipo de 6 pokémon más fuerte y cuál el más débil, considerando su Total y tipo?
Para esto consideraremos los valores obtenidos en Offensive_Power, teniendo en cuenta que cada equipo está conformado por 6 pokemons

In [40]:
# Paso 12. Eleccion del equipo mas fuerte
df_base = df[~df['Name'].str.contains('Mega|Forme|X$|Y$|Black|White|Primal', regex=True)]

top_6_tipos = top_ofensivos['Type 1'].head(6).tolist()

equipo_fuerte = []
for tipo in top_6_tipos:
    pokemon = df_base[df_base['Type 1'] == tipo].nlargest(1, 'Total').iloc[0]
    equipo_fuerte.append(pokemon)

equipo_fuerte_df = pd.DataFrame(equipo_fuerte)[['Name', 'Type 1', 'Type 2', 'Total', 'Attack', 'Sp. Atk']]

print("Equipo más fuerte:\n")
print(equipo_fuerte_df.to_string(index=False))

Equipo más fuerte:

    Name  Type 1 Type 2  Total  Attack  Sp. Atk
Rayquaza  Dragon Flying    680     150      150
   Ho-oh    Fire Flying    680     130      110
 Noivern  Flying Dragon    535      70       97
  Mewtwo Psychic   None    680     110      154
 Yveltal    Dark Flying    680     131      131
  Dialga   Steel Dragon    680     120      150


In [41]:
# Paso 13. Eleccion del equipo mas debil
equipo_debil = []
for tipo in top_6_tipos:
    pokemon = df_base[df_base['Type 1'] == tipo].nsmallest(1, 'Total').iloc[0]
    equipo_debil.append(pokemon)

equipo_debil_df = pd.DataFrame(equipo_debil)[['Name', 'Type 1', 'Type 2', 'Total', 'Attack', 'Sp. Atk']]

print("Equipo más débil (mismos tipos):\n")
print(equipo_debil_df.to_string(index=False))

Equipo más débil (mismos tipos):

     Name  Type 1  Type 2  Total  Attack  Sp. Atk
  Dratini  Dragon    None    300      64       50
   Slugma    Fire    None    250      40       70
   Noibat  Flying  Dragon    245      30       45
    Ralts Psychic   Fairy    198      25       45
Poochyena    Dark    None    220      55       30
   Beldum   Steel Psychic    300      55       35


## Interpretación — Pregunta 4

**Equipo fuerte:** Rayquaza, Ho-oh, Noivern, Mewtwo, Yveltal, Dialga
**Equipo débil:** Dratini, Slugma, Noibat, Ralts, Poochyena, Beldum

La diferencia de Total es devastadora:
- El equipo fuerte promedia 680 puntos por pokémon
- El equipo débil promedia apenas 252 puntos

**Datos curiosos:**
- Dratini y Ralts son las formas bebé de Dragonite y Gardevoir — 
  con suficiente entrenamiento se vuelven monstruos
- Noibat evoluciona a Noivern, que está en el equipo fuerte. 
  ¡Son el mismo pokémon en diferente momento de vida!
- Poochyena tiene menos Attack (55) que Mewtwo tiene de Sp. Atk (154). 
  Literalmente el doble
- Si estos dos equipos se enfrentaran, el débil no sobreviviría 
  ni el primer turno. Ni con steroides.